In [1]:
# ─────────────────────────────────────────────
# Part 1. 라이브러리 임포트 & 데이터 로드
# ─────────────────────────────────────────────
import json
from collections import Counter

# JSON 파일 열기 (with문 쓰면 다 읽고 나서 자동으로 파일 닫힘)
with open(r"C:\lecture\flow\00_data\02_processed\ingredient_merged.json", "r", encoding="utf-8") as f:
    data = json.load(f)  # 리스트 형태로 로드 → data[0]이 성분 1개짜리 딕셔너리

print(f"총 행 수: {len(data)}")          # 87,712개인지 확인
print(f"컬럼 목록: {list(data[0].keys())}")  # 39개 컬럼 확인

총 행 수: 87712
컬럼 목록: ['ingredient_ko', 'ingredient_en', 'pc_rating', 'pc_effect', 'pc_category', 'pc_description', 'coos_function', 'coos_type', 'coos_cas_no', 'coos_country', 'coos_kr_restricted', 'coos_cn_restricted', 'coos_tw_restricted', 'coos_jp_restricted', 'coos_eu_restricted', 'coos_asean_restricted', 'coos_ai_description', 'coos_score', 'coos_data_grade', 'hw_product_id', 'hw_product_name', 'hw_brand_name', 'hw_avg_ratings', 'hw_review_count', 'hw_price', 'hw_consumer_price', 'hw_primary_attr', 'hw_topics_positive', 'hw_topics_negative', 'hw_ingredient_count', 'hw_ingredient_id', 'hw_ewg', 'hw_ewg_data_availability_text', 'hw_is_allergy', 'hw_purpose', 'hw_purposes', 'hw_limitation', 'hw_forbidden', 'hw_category']


In [2]:
# ─────────────────────────────────────────────
# Part 2. 청크 가중치 설정
# ─────────────────────────────────────────────

# 일단 전부 1.0으로 통일 → 추후 실험하면서 조정 예정
CHUNK_WEIGHT = {
    "ewg":        1.0,
    "basic_info": 1.0,
    "expert":     1.0,
    "review":     1.0,
}

# ※ 출처별 가중치(SOURCE_WEIGHT)는 추후 설문 기반 WoE 방식으로 적용 예정 → 여기선 제외

print("가중치 설정 완료")
print(f"청크 가중치: {CHUNK_WEIGHT}")

가중치 설정 완료
청크 가중치: {'ewg': 1.0, 'basic_info': 1.0, 'expert': 1.0, 'review': 1.0}


In [3]:
# ─────────────────────────────────────────────
# Part 3. 청크 변환 함수 정의
# ─────────────────────────────────────────────

def row_to_chunks(row: dict) -> list[dict]:
    chunks = []
    ingredient    = str(row.get("ingredient_ko") or "")
    ingredient_en = str(row.get("ingredient_en") or "")

    base_meta = {
        "ingredient_ko": ingredient,
        "ingredient_en": ingredient_en,
        "coos_score":    row.get("coos_score"),
        "hw_ewg":        row.get("hw_ewg"),
        "is_allergy":    row.get("hw_is_allergy"),
        "pc_rating":     row.get("pc_rating"),
    }

    # ── 청크 1: EWG 등급 ──────────────────────
    ewg_parts = []
    if row.get("coos_score"):
        ewg_parts.append(f"EWG 스코어: {row['coos_score']}")
    if row.get("coos_data_grade"):
        ewg_parts.append(f"데이터 등급: {row['coos_data_grade']}")
    if row.get("hw_ewg"):
        ewg_parts.append(f"화해 EWG: {row['hw_ewg']}")
    if row.get("hw_ewg_data_availability_text"):
        ewg_parts.append(f"EWG 데이터: {row['hw_ewg_data_availability_text']}")

    if ewg_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " / ".join(ewg_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "ewg",
                "chunk_weight": CHUNK_WEIGHT["ewg"],
                "source":       "coos",
            }
        })

    # ── 청크 2: 성분 기본정보 ─────────────────
    # 규제 컬럼 전부 포함 (국내/중국/대만/일본/유럽/아세안)
    basic_parts = []
    for col, label in [
        ("ingredient_en",        "INCI"),
        ("coos_function",        "기능"),
        ("coos_type",            "종류"),
        ("coos_country",         "국가"),
        ("coos_kr_restricted",   "국내규제"),
        ("coos_cn_restricted",   "중국규제"),
        ("coos_tw_restricted",   "대만규제"),
        ("coos_jp_restricted",   "일본규제"),
        ("coos_eu_restricted",   "유럽규제"),
        ("coos_asean_restricted","아세안규제"),
        ("pc_effect",            "효과"),
        ("pc_category",          "분류"),
        ("hw_purpose",           "화장품목적"),
        ("hw_category",          "제품카테고리"),
    ]:
        val = row.get(col)
        if val and str(val) not in ("None", "nan", "규제 내용", "없음"):
            basic_parts.append(f"{label}: {val}")

    if basic_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " / ".join(basic_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "basic_info",
                "chunk_weight": CHUNK_WEIGHT["basic_info"],
                "source":       "coos",
            }
        })

    # ── 청크 3: 전문가 설명 ───────────────────
    expert_parts = []
    if row.get("pc_description"):
        expert_parts.append(f"PC 설명: {row['pc_description']}")
    if row.get("coos_ai_description"):
        expert_parts.append(f"AI 설명: {row['coos_ai_description']}")

    if expert_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " | ".join(expert_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "expert",
                "chunk_weight": CHUNK_WEIGHT["expert"],
                "source":       "pc",
            }
        })

    # ── 청크 4: 리뷰·피부반응 ────────────────
    review_parts = []
    if row.get("hw_topics_positive"):
        review_parts.append(f"긍정 반응: {row['hw_topics_positive']}")
    if row.get("hw_topics_negative"):
        review_parts.append(f"부정 반응: {row['hw_topics_negative']}")
    if row.get("hw_primary_attr"):
        review_parts.append(f"주요속성: {row['hw_primary_attr']}")

    if review_parts:
        chunks.append({
            "page_content": f"[{ingredient}] " + " / ".join(review_parts),
            "metadata": {
                **base_meta,
                "chunk_type":   "review",
                "chunk_weight": CHUNK_WEIGHT["review"],
                "source":       "hw",
            }
        })

    return chunks

print("청크 변환 함수 정의 완료")

청크 변환 함수 정의 완료


In [4]:
# ─────────────────────────────────────────────
# Part 3-1. 제품 추천 정보 추가 ( 제품 추천은 RAG가 아니라 별도 DB로 처리)
# ─────────────────────────────────────────────

import pandas as pd
import json

with open(r"C:\lecture\flow\00_data\02_processed\ingredient_merged.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 제품 추천용 DB만 따로 추출
product_cols = [
    "ingredient_ko",
    "ingredient_en",
    "hw_product_id",
    "hw_product_name",
    "hw_brand_name",
    "hw_avg_ratings",
    "hw_review_count",
    "hw_price",
    "hw_consumer_price",
    "hw_primary_attr",
    "hw_category",
    "hw_is_allergy",
    "hw_limitation",
    "hw_forbidden",
]

df_product = pd.DataFrame(data)[product_cols]

# 화해 데이터 없는 행 제거
df_product = df_product.dropna(subset=["hw_product_name"])

print(f"제품 DB 행 수: {len(df_product)}")
print(df_product.head())

# CSV로 저장
df_product.to_csv(
    r"C:\lecture\flow\00_data\02_processed\product_db.csv",
    index=False,
    encoding="utf-8-sig"
)
print("제품 DB 저장 완료")

제품 DB 행 수: 71139
   ingredient_ko   ingredient_en  hw_product_id  \
6      1,2-헥산다이올  1,2-Hexanediol        67749.0   
7      1,2-헥산다이올  1,2-Hexanediol        77283.0   
8      1,2-헥산다이올  1,2-Hexanediol        63363.0   
9      1,2-헥산다이올  1,2-Hexanediol        74213.0   
10     1,2-헥산다이올  1,2-Hexanediol        71372.0   

                               hw_product_name hw_brand_name  hw_avg_ratings  \
6                          블레미쉬 밤 [SPF26/PA++]         파파레서피            4.20   
7   어성초 진정 블레미쉬 커버 선비비 뉴트럴 베이지 [SPF50+/PA++++]            구달            4.60   
8                                 펩타인 바이옴 BB크림         스와니코코            4.29   
9                      인텐시브 세럼 비비 [SPF34/PA++]          셀바이셀            3.92   
10         셀룸 아마이드 BB 크림 [SPF50+/PA++++] [내추럴]          유리피부            4.37   

    hw_review_count  hw_price  hw_consumer_price hw_primary_attr hw_category  \
6             396.0   25000.0            25000.0              새틴     BB/CC크림   
7              20.0   17600.0      

In [5]:
# ─────────────────────────────────────────────
# Part 4. 전체 데이터 청크 변환 실행
# ─────────────────────────────────────────────

all_chunks = []

for row in data:
    # 성분 1개 → 청크 리스트로 변환 후 all_chunks에 펼쳐서 추가
    # extend() : 리스트를 펼쳐서 붙임 (append()는 리스트째로 넣어서 중첩됨)
    all_chunks.extend(row_to_chunks(row))

print(f"총 청크 수: {len(all_chunks)}")

# 청크 타입별 몇 개씩 만들어졌는지 확인
type_counts = Counter(c["metadata"]["chunk_type"] for c in all_chunks)
print(f"청크 타입별 분포: {dict(type_counts)}")

# 각 타입별 샘플 1개씩 출력해서 내용 확인
for chunk_type in ["ewg", "basic_info", "expert", "review"]:
    sample = next((c for c in all_chunks if c["metadata"]["chunk_type"] == chunk_type), None)
    if sample:
        print(f"\n{'='*50}")
        print(f"[{chunk_type} 샘플]")
        print(json.dumps(sample, ensure_ascii=False, indent=2))

총 청크 수: 306836
청크 타입별 분포: {'ewg': 86169, 'basic_info': 87712, 'expert': 61816, 'review': 71139}

[ewg 샘플]
{
  "page_content": "[(20S)-프로토파낙사다이올] EWG 스코어: 0 / 데이터 등급: 0",
  "metadata": {
    "ingredient_ko": "(20S)-프로토파낙사다이올",
    "ingredient_en": "(20S)-Protopanaxadiol",
    "coos_score": "0",
    "hw_ewg": null,
    "is_allergy": null,
    "pc_rating": null,
    "chunk_type": "ewg",
    "chunk_weight": 1.0,
    "source": "coos"
  }
}

[basic_info 샘플]
{
  "page_content": "[(20S)-프로토파낙사다이올] INCI: (20S)-Protopanaxadiol / 기능: AI Anti-aging/Wrinkle Care AI Antioxidant 식 피부보호제, 피부컨디셔닝제(유연제), 피부컨디셔닝제(보습제) / 종류: AI Actives Protopanaxadiol is a ginsenoside with bioactive properties, making it an active ingredient in formulations.",
  "metadata": {
    "ingredient_ko": "(20S)-프로토파낙사다이올",
    "ingredient_en": "(20S)-Protopanaxadiol",
    "coos_score": "0",
    "hw_ewg": null,
    "is_allergy": null,
    "pc_rating": null,
    "chunk_type": "basic_info",
    "chunk_weight": 1.0,
    "source": "co

In [6]:
# ─────────────────────────────────────────────
# Part 5. 제품 DB 별도 저장
# ─────────────────────────────────────────────
import pandas as pd

# 제품 추천에 필요한 컬럼만 추출
product_cols = [
    "ingredient_ko",
    "ingredient_en",
    "hw_product_id",
    "hw_product_name",
    "hw_brand_name",
    "hw_avg_ratings",
    "hw_review_count",
    "hw_price",
    "hw_consumer_price",
    "hw_primary_attr",
    "hw_category",
    "hw_is_allergy",
    "hw_limitation",
    "hw_forbidden",
]

df_product = pd.DataFrame(data)[product_cols]

# 화해 데이터 없는 행 제거 (hw_product_name이 null인 행)
df_product = df_product.dropna(subset=["hw_product_name"])

print(f"제품 DB 행 수: {len(df_product)}")
print(f"\n[샘플 3개]")
print(df_product.head(3).to_string())

# CSV로 저장
PRODUCT_DB_PATH = r"C:\lecture\flow\00_data\02_processed\product_db.csv"
df_product.to_csv(PRODUCT_DB_PATH, index=False, encoding="utf-8-sig")
print(f"\n제품 DB 저장 완료: {PRODUCT_DB_PATH}")

제품 DB 행 수: 71139

[샘플 3개]
  ingredient_ko   ingredient_en  hw_product_id                             hw_product_name hw_brand_name  hw_avg_ratings  hw_review_count  hw_price  hw_consumer_price hw_primary_attr hw_category hw_is_allergy hw_limitation hw_forbidden
6     1,2-헥산다이올  1,2-Hexanediol        67749.0                         블레미쉬 밤 [SPF26/PA++]         파파레서피            4.20            396.0   25000.0            25000.0              새틴     BB/CC크림         False         해당 없음        해당 없음
7     1,2-헥산다이올  1,2-Hexanediol        77283.0  어성초 진정 블레미쉬 커버 선비비 뉴트럴 베이지 [SPF50+/PA++++]            구달            4.60             20.0   17600.0            22000.0              새틴     BB/CC크림         False         해당 없음        해당 없음
8     1,2-헥산다이올  1,2-Hexanediol        63363.0                                펩타인 바이옴 BB크림         스와니코코            4.29             48.0   35000.0            45000.0              새틴     BB/CC크림         False         해당 없음        해당 없음

제품 DB 저장 완료: C:\lecture\flow\

In [8]:
# ─────────────────────────────────────────────
# Part 6. 청크 파일 저장
# ─────────────────────────────────────────────

OUTPUT_PATH = r"C:\lecture\flow\00_data\02_processed\ingredient_chunks.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        all_chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"청크 저장 완료: {OUTPUT_PATH}")
print(f"총 청크 수: {len(all_chunks)}")

청크 저장 완료: C:\lecture\flow\00_data\02_processed\ingredient_chunks.json
총 청크 수: 306836
